# Pandas ExtensionArray API for Arkouda Demo

In [ ]:
import pandas as pd
from pathlib import Path
import arkouda as ak
from arkouda.pandas import extension

ak.connect()

# Read in the data

### Intrusion Detection Evaluation Dataset (CIC-IDS2017)
* Canadian Institute for Cybersecurity
* Netflow examples
* [CIC-IDS2017 Dataset](https://www.unb.ca/cic/datasets/ids-2017.html)

### Bug: 
* https://github.com/Bears-R-Us/arkouda/issues?q=is%3Aissue%20state%3Aopen%20CSV%20label%3Abug
* https://github.com/Bears-R-Us/arkouda/issues/5429

In [ ]:

df = ak.read_csv("/home/amandapotts/git/arkouda-netflow-demo/data/TrafficLabelling/M*")


### Use Pandas for IO

In [ ]:

def _read_csv_robust(path: Path) -> pd.DataFrame:
    # Try common encodings seen in CICIDS2017 exports
    for enc in ("utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    # If all fail, raise the last error by explicitly reading as bytes is overkill here
    raise UnicodeDecodeError("utf-8", b"", 0, 1, f"Failed to decode {path} with utf-8/cp1252/latin-1")


def load_trafficlabelling(data_dir: str | Path) -> pd.DataFrame:
    data_dir = Path(data_dir)
    csv_files = sorted(data_dir.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")

    dfs: list[pd.DataFrame] = []
    for csv_file in csv_files:
        print(f"Reading {csv_file.name}")
        df = _read_csv_robust(csv_file)

        # CICIDS2017 often has whitespace in column names
        df.columns = df.columns.str.strip()

        df["source_file"] = csv_file.name
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


In [ ]:
traffic_dir = Path("/home/amandapotts/git/arkouda-netflow-demo/data/TrafficLabelling")
df = load_trafficlabelling(traffic_dir)

In [ ]:
df.head()

###  Convert to Arkouda-backed columns (using the DataFrame accessor)

In [ ]:
df["Destination IP"].values

In [ ]:
ak_df = df.ak.to_ak()

In [ ]:
type(ak_df)

In [ ]:
ak_df["Destination IP"].values

# Exploratory Data Analysis

In [ ]:
print(f"Rows: {ak_df.shape[0]:,}")
print(f"Columns: {ak_df.shape[1]}")

In [ ]:
ak_df.head()

In [ ]:
type(ak_df.head()["Flow ID"].values)

### Column Access

In [ ]:
print("\n=== Source Port sample ===")
ak_df["Source Port"][:10]

In [ ]:
type(ak_df["Source Port"][:10].values)

### Filtering

#### TCP flows only (Protocol == 6)

In [ ]:
tcp = ak_df[ak_df["Protocol"] == 6]
print(f"TCP flows: {tcp.shape[0]:,}")

In [ ]:
tcp

#### Top-k style aggregations

In [ ]:
destination_port_series = tcp["Destination Port"].value_counts().sort_values(ascending=False).head(10)
destination_port_series

In [ ]:
dest_ports, counts = (
    destination_port_series.index,
    destination_port_series.values,
)

In [ ]:
dest_ports

In [ ]:
type(dest_ports)

In [ ]:
dest_ports.values

In [ ]:
counts

#### GroupBy operations

In [ ]:
counts = tcp.groupby("Destination Port").size()
counts

In [ ]:
type(counts.values)

#### GroupBy using DataFrame accessor

In [ ]:
dest_ports, counts = tcp["Destination Port"].ak.groupby().size()

In [ ]:
dest_ports

In [ ]:
type(dest_ports)

In [ ]:
top = counts.argsort(ascending=False)[:10]
for p, c in zip(dest_ports[top].to_ndarray(), counts[top].to_ndarray()):
    print(f"Port {int(p):>5}: {int(c):>10} flows")

#### Argsort

In [ ]:
print("Data Obj Type:   " + str(type(destination_port_series)))
print("Values Obj Type: " + str(type(destination_port_series.values)))
print("Index Obj Type:  " + str(type(destination_port_series.index.values)))

In [ ]:
destination_port_series.argsort()

In [ ]:
type((destination_port_series.argsort().values))

In [ ]:
destination_port_series.ak.argsort()

In [ ]:
type((destination_port_series.ak.argsort().values))

#### Numeric Summaries

In [ ]:
durations = tcp["Flow Duration"]
print(
    f"min={durations.min():.1f}, "
    f"mean={durations.mean():.1f}, "
    f"max={durations.max():.1f}"
)

#### Multi-column Summaries

In [ ]:
gb = tcp["Destination Port"].ak.groupby()

In [ ]:
mean_bytes = gb.mean(
    (tcp["Total Length of Fwd Packets"]
    + tcp["Total Length of Bwd Packets"])
)

In [ ]:
mean_bytes

In [ ]:
ports, avg_bytes = mean_bytes
top = avg_bytes.argsort(ascending=False)[:10]

In [ ]:
for p, b in zip(ports[top].to_ndarray(), avg_bytes[top].to_ndarray()):
    print(f"Port {int(p):>5}: mean bytes/flow = {b:,.1f}")

#### Threshold-based filtering (common NetFlow pattern)

In [ ]:
total_bytes = (
    tcp["Total Length of Fwd Packets"] +
    tcp["Total Length of Bwd Packets"]
)

In [ ]:
print("\n=== Large flows (bytes > 1e5) ===")
large = tcp[total_bytes > 1e5]
print(f"Large flows: {large.shape[0]:,}")

#### Apply

In [ ]:
ak_df["Flow Duration"]

In [ ]:
pd_apply = ak_df["Flow Duration"].apply(lambda x: x /1000)
pd_apply

In [ ]:
type(pd_apply)

In [ ]:
type(pd_apply.values)

In [ ]:
ak_apply = ak_df["Flow Duration"].ak.apply(lambda x: x /1000)
ak_apply

In [ ]:
type(ak_apply)

In [ ]:
type(ak_apply.values)

### Thank You!

In [ ]:
ak.shutdown()